# Create Field Collection Layer

Creates a **hosted feature layer** on ArcGIS Online for field data collection via **Field Maps**.

## Workflow

1. **Select a schema** from the `schemas/` package (e.g. Plant Identification)
2. **Optionally seed geometries** from an existing layer (e.g. AI-detected tree circles)
3. **Create the service** with:
   - Schema fields + approval fields (`review_status`, `reviewed_by`, `review_date`, `revision_notes`)
   - Editor tracking (created/edited by/date)
   - Attachments enabled for photo capture
   - Web Mercator (102100) for proper Shape__Area/Length in meters

Features start with `review_status = NULL`. An **Arcade expression** on the Collector view sets it to `'Pending Review'` on every save. The feature then progresses:

```
NULL → Pending Review → In Review → Approved / Rejected / Needs Revision
                                                              ↓
                                                     back to Collector (NULL + Needs Revision)
```

## Prerequisites

- ArcGIS Online organizational account with Creator or higher license
- `arcgis` Python package ≥ 2.x
- (Optional) Output from `detect-objects-deep-learning.ipynb` as a geometry source

## Step 1 – Connect to ArcGIS Online

In [ ]:
import getpass, json, urllib3
from pathlib import Path
from arcgis.gis import GIS

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ── Staging toggle ─────────────────────────────────────────────────────
# Set to True for test runs — uses a separate registry and adds
# " - STAGING" to all item titles so production items are never touched.
STAGING = True
# ───────────────────────────────────────────────────────────────────────

_suffix = " - STAGING" if STAGING else ""

username = input("ArcGIS Online username: ")
password = getpass.getpass("ArcGIS Online password: ")
gis = GIS("https://www.arcgis.com", username, password, verify_cert=False)
print(f"Signed in as: {gis.users.me.username} ({gis.properties.portalName})")
if STAGING:
    print(f"⚠️  STAGING MODE — items will be titled with '{_suffix}'")

Setting `verify_cert` to False is a security risk, use at your own risk.


Signed in as: ralouta.aiddev (ArcGIS Online)


## Step 2 – Select a Data Collection Schema

Choose from the available schemas in the `schemas/` package. Each schema defines the fields, domains, and renderer for a specific data-collection use case. New schemas can be added by creating a module in `schemas/` and registering it in `schemas/__init__.py`.

In [4]:
import sys, os, importlib
import ipywidgets as widgets
from IPython.display import display

# Ensure the repo root is on sys.path
_repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import schemas
importlib.reload(schemas)

# Schema selection dropdown (re-create on every run to avoid stacked observers)
_schema_dropdown = widgets.Dropdown(
    options=list(schemas.SCHEMAS.keys()),
    value=list(schemas.SCHEMAS.keys())[0],
    description="Schema:",
    layout=widgets.Layout(width="350px"),
)

_schema_desc = widgets.HTML()
_schema_fields_out = widgets.Output()


def _on_schema_change(change):
    mod_path = schemas.SCHEMAS[change["new"]]
    mod = importlib.import_module(mod_path)
    importlib.reload(mod)
    _schema_desc.value = f"<i>{mod.SCHEMA_INFO['description']}</i>"
    _schema_fields_out.clear_output(wait=True)
    with _schema_fields_out:
        for f in mod.FIELDS:
            print(f"  {f['alias']:30s}  ({f['type'].replace('esriFieldType', '')})")


_schema_dropdown.observe(_on_schema_change, names="value")
_on_schema_change({"new": _schema_dropdown.value})  # trigger initial load

display(_schema_dropdown, _schema_desc)
print("\nSchema fields:")
display(_schema_fields_out)
print("\nRun the next cell after selecting your schema.")

Dropdown(description='Schema:', layout=Layout(width='350px'), options=('Plant Identification',), value='Plant …

HTML(value='<i>Horticultural plant identification observations with photo attachments. Observers capture a pho…


Schema fields:


Output()


Run the next cell after selecting your schema.


In [5]:
# Load the selected schema module
mod_path = schemas.SCHEMAS[_schema_dropdown.value]
schema_mod = importlib.import_module(mod_path)
importlib.reload(schema_mod)

schema_label = _schema_dropdown.value
schema_fields = schema_mod.FIELDS
schema_domains = schema_mod.DOMAINS
schema_meta = schema_mod.LAYER_META

print(f"Loaded schema: {schema_label}")
print(f"  Fields:  {len(schema_fields)}")
print(f"  Domains: {len(schema_domains)}")
print(f"  Geometry: {schema_meta['geometryType']}")

Loaded schema: Plant Identification
  Fields:  10
  Domains: 2
  Geometry: esriGeometryPoint


## Step 3 – Optional Geometry Source

If you have an existing feature layer (e.g. output from the **Detect Objects** notebook), enter its **Item ID** below. The geometries will be copied into the new layer so that field observers see pre-placed features on the map.

Leave blank to create an empty layer (observers will create new point features in Field Maps).

In [6]:
geometry_source_id = input("Geometry source Item ID (leave blank for empty layer): ").strip()

source_features = None
source_sr = None
source_geom_type = None

if geometry_source_id:
    source_item = gis.content.get(geometry_source_id)
    if source_item is None:
        raise ValueError(f"Item '{geometry_source_id}' not found. Check the ID and your permissions.")

    source_layer = source_item.layers[0]
    # Reproject to Web Mercator (102100) to match the target service
    source_fs = source_layer.query(where="1=1", out_sr=102100)
    source_features = source_fs.features
    source_sr = {"wkid": 102100, "latestWkid": 3857}
    source_geom_type = source_layer.properties.geometryType

    print(f"Loaded: {source_item.title}")
    print(f"  Features:  {len(source_features)}")
    print(f"  Geometry:  {source_geom_type}")
    print(f"  Reprojected to WKID 102100 (Web Mercator)")
else:
    print("No geometry source — layer will be created empty.")

Loaded: Trees SAM3 AOI1 QAQC
  Features:  1310
  Geometry:  esriGeometryPoint
  Reprojected to WKID 102100 (Web Mercator)


In [7]:
import ipywidgets as widgets
from IPython.display import display

if source_features:
    # Build field list from source layer (exclude system/geometry fields)
    _SYS = {"objectid", "fid", "globalid", "shape", "shape_length", "shape_area",
            "shape__length", "shape__area", "st_length(shape)", "st_area(shape)"}
    _src_fields = [
        f for f in source_layer.properties.fields
        if f["name"].lower() not in _SYS
    ]
    _field_names = [f["name"] for f in _src_fields]

    _filter_enabled_w = widgets.Checkbox(value=False, description="Enable filter", indent=False)

    _field_w = widgets.Dropdown(
        options=_field_names,
        description="Field:",
        layout=widgets.Layout(width="350px"),
    )

    _op_w = widgets.Dropdown(
        options=[
            ("equals (=)", "="),
            ("not equal (<>)", "<>"),
            ("less than (<)", "<"),
            ("less or equal (<=)", "<="),
            ("greater than (>)", ">"),
            ("greater or equal (>=)", ">="),
            ("LIKE", "LIKE"),
            ("IS NULL", "IS NULL"),
            ("IS NOT NULL", "IS NOT NULL"),
        ],
        value="=",
        description="Operator:",
        layout=widgets.Layout(width="350px"),
    )

    _val_w = widgets.Text(
        value="",
        placeholder="Value (ignored for IS NULL / IS NOT NULL)",
        description="Value:",
        layout=widgets.Layout(width="400px"),
    )

    _filter_box = widgets.VBox([_field_w, _op_w, _val_w])
    _filter_box.layout.display = "none"

    def _toggle_filter(change):
        _filter_box.layout.display = "" if change["new"] else "none"
    _filter_enabled_w.observe(_toggle_filter, names="value")

    print("Optional: filter source features before seeding.")
    display(_filter_enabled_w)
    display(_filter_box)
    print("Run the next cell to apply the filter.")
else:
    _filter_enabled_w = None
    print("No source features — filter step skipped.")

Optional: filter source features before seeding.


Checkbox(value=False, description='Enable filter', indent=False)

Run the next cell to apply the filter.


In [8]:
# Apply the filter to source features (if enabled)
if source_features and _filter_enabled_w and _filter_enabled_w.value:
    _fname = _field_w.value
    _op = _op_w.value
    _val = _val_w.value.strip()

    _before = len(source_features)

    def _matches(feat):
        v = (feat.attributes or {}).get(_fname)
        if _op == "IS NULL":
            return v is None
        if _op == "IS NOT NULL":
            return v is not None
        if v is None:
            return False
        # Try numeric comparison first
        try:
            v_num = float(v)
            val_num = float(_val)
            if _op == "=":   return v_num == val_num
            if _op == "<>":  return v_num != val_num
            if _op == "<":   return v_num < val_num
            if _op == "<=":  return v_num <= val_num
            if _op == ">":   return v_num > val_num
            if _op == ">=":  return v_num >= val_num
        except (ValueError, TypeError):
            pass
        # String comparison
        v_str = str(v)
        if _op == "=":    return v_str == _val
        if _op == "<>":   return v_str != _val
        if _op == "LIKE":
            import re
            pattern = "^" + re.escape(_val).replace(r"\%", ".*").replace(r"\_", ".") + "$"
            return bool(re.match(pattern, v_str, re.IGNORECASE))
        return False

    source_features = [f for f in source_features if _matches(f)]
    print(f"Filter: {_fname} {_op} {_val if _op not in ('IS NULL', 'IS NOT NULL') else ''}")
    print(f"  Before: {_before}  →  After: {len(source_features)}")
else:
    if source_features:
        print(f"No filter applied — using all {len(source_features)} source features.")
    else:
        print("No source features — nothing to filter.")

Filter: det_type = circular
  Before: 1310  →  After: 1000


## Step 4 – Configure & Create the Hosted Feature Layer

The layer is built with:
- **Point geometry** — trees are represented as trunk-base points (polygon sources are converted to centroids during seeding)
- **Schema fields** from the selected schema
- **Approval workflow** field (`review_status`) with coded-value domain
- **Editor tracking** fields (created/edited by/date)
- **Attachments enabled** for photo capture in Field Maps

In [ ]:
from datetime import datetime
from arcgis.features import FeatureLayerCollection
import ipywidgets as widgets
from IPython.display import display

# ── Layer title ────────────────────────────────────────────────────────
_default_title = f"{schema_label} - Field Collection{_suffix}"
layer_title = input(f"Layer title [{_default_title}]: ").strip() or _default_title

# ── Folder selection ──────────────────────────────────────────────────
# Fetch existing folders from the user's content
_user_folders = gis.users.me.folders
_existing_folder_names = [getattr(f, 'name', None) or getattr(f, 'title', str(f)) for f in _user_folders]

_folder_mode_w = widgets.Dropdown(
    options=["Root (no folder)", "Existing folder", "Create new folder"],
    value="Root (no folder)",
    description="Folder:",
    layout=widgets.Layout(width="350px"),
)

_existing_folder_w = widgets.Dropdown(
    options=_existing_folder_names if _existing_folder_names else ["(no folders)"],
    description="Select:",
    layout=widgets.Layout(width="400px"),
)

_new_folder_w = widgets.Text(
    value="",
    placeholder="Enter new folder name",
    description="Name:",
    layout=widgets.Layout(width="400px"),
)

_existing_folder_box = widgets.HBox([_existing_folder_w])
_new_folder_box = widgets.HBox([_new_folder_w])
_existing_folder_box.layout.display = "none"
_new_folder_box.layout.display = "none"

def _on_folder_mode_change(change):
    _existing_folder_box.layout.display = "none"
    _new_folder_box.layout.display = "none"
    if change["new"] == "Existing folder":
        _existing_folder_box.layout.display = ""
    elif change["new"] == "Create new folder":
        _new_folder_box.layout.display = ""

_folder_mode_w.observe(_on_folder_mode_change, names="value")

print(f"Layer title: {layer_title}")
print(f"Existing folders: {_existing_folder_names if _existing_folder_names else '(none)'}\n")
print("Select where to store the layer:")
display(widgets.VBox([_folder_mode_w, _existing_folder_box, _new_folder_box]))
print("\nThen run the next cell to continue.")

# ── Determine geometry type ────────────────────────────────────────────
# Always Point — trees are represented as point features (trunk base).
# Polygon source features (from DL detections) are converted to centroids
# during the seed step.
geom_type = "esriGeometryPoint"
if source_geom_type and source_geom_type != "esriGeometryPoint":
    print(f"Source is {source_geom_type} — will convert to points (centroids) during seeding.")
print(f"\nGeometry type: {geom_type}")

# ── Approval workflow domain ──────────────────────────────────────────
review_status_domain = {
    "type": "codedValue",
    "name": "ReviewStatusDomain",
    "description": "Approval workflow status for AI-enriched records.",
    "fieldType": "esriFieldTypeString",
    "codedValues": [
        {"name": "Pending Review",  "code": "Pending Review"},
        {"name": "In Review",       "code": "In Review"},
        {"name": "Approved",        "code": "Approved"},
        {"name": "Rejected",        "code": "Rejected"},
        {"name": "Needs Revision",  "code": "Needs Revision"},
    ],
    "mergePolicy": "esriMPTDefaultValue",
    "splitPolicy": "esriSPTDuplicate",
}

# ── System + approval + editor tracking fields ────────────────────────
system_fields = [
    {"name": "OBJECTID", "type": "esriFieldTypeOID", "alias": "OBJECTID",
     "nullable": False, "editable": False},
    {"name": "GlobalID", "type": "esriFieldTypeGlobalID", "alias": "GlobalID",
     "nullable": False, "editable": False},
]

approval_fields = [
    {"name": "review_status", "type": "esriFieldTypeString", "alias": "Review Status",
     "length": 30, "nullable": True, "editable": True,
     "defaultValue": None,
     "description": "Approval workflow status. NULL on creation; Arcade sets to 'Pending Review' on save.",
     "domain": review_status_domain},
    {"name": "reviewed_by", "type": "esriFieldTypeString", "alias": "Reviewed By",
     "length": 255, "nullable": True, "editable": True,
     "description": "Name of the reviewer who approved/rejected/sent back the record."},
    {"name": "review_date", "type": "esriFieldTypeDate", "alias": "Review Date",
     "nullable": True, "editable": True,
     "description": "Date/time the review decision was made."},
    {"name": "revision_notes", "type": "esriFieldTypeString", "alias": "Revision Notes",
     "length": 500, "nullable": True, "editable": True,
     "description": "Notes from the reviewer explaining what needs to be fixed (Needs Revision)."},
]

editor_fields = [
    {"name": "created_user", "type": "esriFieldTypeString", "alias": "Created By",
     "length": 255, "nullable": True, "editable": False},
    {"name": "created_date", "type": "esriFieldTypeDate", "alias": "Created Date",
     "nullable": True, "editable": False},
    {"name": "last_edited_user", "type": "esriFieldTypeString", "alias": "Last Edited By",
     "length": 255, "nullable": True, "editable": False},
    {"name": "last_edited_date", "type": "esriFieldTypeDate", "alias": "Last Edited Date",
     "nullable": True, "editable": False},
]

# ── Assemble the full field list ──────────────────────────────────────
all_fields = system_fields + schema_fields + approval_fields + editor_fields

# ── Drawing tool for templates ────────────────────────────────────────
drawing_tool_map = {
    "esriGeometryPoint": "esriFeatureEditToolPoint",
    "esriGeometryPolygon": "esriFeatureEditToolPolygon",
    "esriGeometryPolyline": "esriFeatureEditToolLine",
}

# Merge prototype attributes from schema templates
# review_status starts as NULL; Arcade on the Collector view sets it to 'Pending Review' on save
_proto_attrs = {}
if schema_meta.get("templates"):
    _proto_attrs.update(schema_meta["templates"][0].get("prototype", {}).get("attributes", {}))

# ── Layer definition ──────────────────────────────────────────────────
layer_def = {
    "name": schema_meta["name"],
    "type": "Feature Layer",
    "description": schema_meta["description"],
    "geometryType": geom_type,
    "objectIdField": "OBJECTID",
    "globalIdField": "GlobalID",
    "hasAttachments": True,
    "hasStaticData": False,
    "hasM": False,
    "hasZ": False,
    "allowGeometryUpdates": True,
    "supportsApplyEditsWithGlobalIds": True,
    "capabilities": "Create,Query,Update,Delete,Editing,Uploads,Sync,Extract",
    "fields": all_fields,
    "types": [],
    "templates": [
        {
            "name": f"New {schema_label}",
            "description": f"Default template for {schema_label} collection.",
            "drawingTool": drawing_tool_map.get(geom_type, "esriFeatureEditToolPoint"),
            "prototype": {"attributes": _proto_attrs},
        }
    ],
    "editFieldsInfo": {
        "creationDateField": "created_date",
        "creatorField": "created_user",
        "editDateField": "last_edited_date",
        "editorField": "last_edited_user",
    },
    "drawingInfo": schema_meta.get("drawingInfo", {
        "renderer": {
            "type": "simple",
            "symbol": {
                "type": "esriSMS",
                "style": "esriSMSCircle",
                "color": [56, 168, 0, 180],
                "size": 8,
                "outline": {"color": [0, 0, 0, 255], "width": 0.75},
            },
        }
    }),
}

print(f"\nLayer definition ready:")
print(f"  Name:       {layer_def['name']}")
print(f"  Geometry:   {layer_def['geometryType']}")
print(f"  Fields:     {len(all_fields)} ({len(schema_fields)} schema + {len(approval_fields)} approval + {len(editor_fields)} editor + {len(system_fields)} system)")
print(f"  Attachments: {layer_def['hasAttachments']}")
print(f"  Approval:   review_status defaults to NULL (Arcade sets on save)")

Layer title: Plant Identification - AI
Existing folders: ['Root Folder', '3D Basemaps', 'ACLED', 'AFG - Snow Detection', 'AGE - EsriAidDev', 'age-int-distributed-reference', 'agol_monitoring', 'Agriculture Census Ethiopia', 'Agriculture-Portal-Lebanon', 'ai workshop 2026', 'AI_GENERATED_CONTENT', 'AJenkins Content', 'Arcade Assistant Demo', 'ArcGIS Maps SDK for JavaScript', 'ArcGIS Server PipeLine Demonstration', 'ArcGIS Web Mapping Workshop', 'ArcGISAIComponents', 'AUB-Demos', 'Beirut', 'Cantabria Biomass', 'Car detection - Esri Spain', 'Change Detection Sentinel', 'City Explorer', 'cloned_dashboards', 'Cows-DL', 'cows_dl', 'DataImpacta', 'datapipeline_ICC', 'Days-At-The-Office', 'Default folder', 'Democratic Republic of the Congo', 'Dl Bare Agriculture Lands', 'DLPKs', 'Doezastraat 12-A', 'ecs_aiddev_test', 'EsriAidDev - AGE-Int', 'extracted_data', 'FAO - Sudan', 'FAO Spatial Analysis', 'FAO-Tile-Cache', 'FAOSS', 'fugro-genaitoolkit', 'G.AI.T Tools', 'GeoAI Demo', 'GeONG Demo', 'GICH


Then run the next cell to continue.

Geometry type: esriGeometryPoint

Layer definition ready:
  Name:       Plant_Observations
  Geometry:   esriGeometryPoint
  Fields:     20 (10 schema + 4 approval + 4 editor + 2 system)
  Attachments: True
  Approval:   review_status defaults to NULL (Arcade sets on save)


In [10]:
import re, time

# ── Resolve folder from widget selection ───────────────────────────────
folder_mode = _folder_mode_w.value
target_folder = None  # None = root

if folder_mode == "Existing folder":
    _sel = _existing_folder_w.value
    if _sel and _sel != "(no folders)":
        target_folder = _sel
    print(f"Folder: {target_folder or '(root)'}")
elif folder_mode == "Create new folder":
    _new_name = _new_folder_w.value.strip()
    if not _new_name:
        raise ValueError("Please enter a folder name in the widget above.")
    # Create the folder if it doesn't already exist
    if _new_name not in [getattr(f, 'name', None) or getattr(f, 'title', str(f)) for f in gis.users.me.folders]:
        gis.content.create_folder(_new_name)
        print(f"Created new folder: {_new_name}")
    else:
        print(f"Folder already exists: {_new_name}")
    target_folder = _new_name
else:
    print("Folder: (root)")

# ── Create the hosted feature service ──────────────────────────────────
# Use Web Mercator (102100) so Shape__Area / Shape__Length are in meters
timestamp = datetime.now().strftime("%Y%m%d%H%M")
_svc_name = re.sub(r"[^0-9A-Za-z_]+", "_", layer_title).strip("_")
service_name = f"{_svc_name}_{timestamp}"

create_params = {
    "name": service_name,
    "serviceDescription": schema_meta["description"],
    "hasStaticData": False,
    "maxRecordCount": 2000,
    "supportedQueryFormats": "JSON",
    "capabilities": "Create,Query,Update,Delete,Editing,Uploads,Sync,Extract",
    "description": f"{schema_label} field data collection layer with photo attachments and approval workflow.",
    "copyrightText": "",
    "spatialReference": {"wkid": 102100, "latestWkid": 3857},
    "initialExtent": {
        "xmin": -20037508.34, "ymin": -20037508.34,
        "xmax": 20037508.34, "ymax": 20037508.34,
        "spatialReference": {"wkid": 102100, "latestWkid": 3857},
    },
    "allowGeometryUpdates": True,
    "units": "esriMeters",
    "xssPreventionInfo": {
        "xssPreventionEnabled": True,
        "xssPreventionRule": "InputOnly",
        "xssInputRule": "rejectInvalid",
    },
}

print(f"Creating service: {service_name}")
print(f"  Spatial reference: Web Mercator (102100) — Shape__Area/Length in meters")
flc_item = gis.content.create_service(
    name=service_name,
    create_params=create_params,
    folder=target_folder,
    item_properties={
        "title": layer_title,
        "tags": f"{schema_label}, field-collection, attachments, AI, approval-workflow",
        "snippet": f"{schema_label} field data collection with photo attachments and AI review workflow.",
        "description": (
            f"Hosted feature layer for {schema_label} data collection. "
            f"Designed for Field Maps with photo attachments and an AI-powered approval workflow. "
            f"Each feature defaults to 'Pending Review' status."
        ),
    },
)
print(f"Created: {flc_item.title}  (Item ID: {flc_item.id})")
if target_folder:
    print(f"Folder:  {target_folder}")

# Push the layer schema
flc = FeatureLayerCollection.fromitem(flc_item)
add_result = flc.manager.add_to_definition({"layers": [layer_def]})

# Validate the result
if isinstance(add_result, dict) and not add_result.get("success", True):
    raise RuntimeError(f"add_to_definition failed: {add_result}")
print(f"Schema applied: {add_result}")

# Enable editor tracking at the service level so created_user, created_date,
# last_edited_user, and last_edited_date are auto-populated by ArcGIS Online
# based on whoever is running the code (the authenticated GIS user).
flc.manager.update_definition({
    "editorTrackingInfo": {
        "enableEditorTracking": True,
        "enableOwnershipAccessControl": False,
        "allowOthersToQuery": True,
        "allowOthersToUpdate": True,
        "allowOthersToDelete": False,
        "allowAnonymousToQuery": True,
        "allowAnonymousToUpdate": False,
        "allowAnonymousToDelete": False,
    }
})
print("Editor tracking enabled — created/edited fields auto-populated by the server")

# Wait for ArcGIS Online to propagate the layer definition
print("Waiting for service propagation...")
time.sleep(8)

# Refresh and verify
flc_item = gis.content.get(flc_item.id)
if not flc_item.layers:
    raise RuntimeError("Service has no layers after add_to_definition — check the layer_def for errors.")
target_layer = flc_item.layers[0]

# Verify the layer is queryable before proceeding
_test = target_layer.query(where="1=1", return_count_only=True)
print(f"Layer verified — {_test} existing features")
print(f"\nLayer URL: {target_layer.url}")
print(f"Fields:    {[f['name'] for f in target_layer.properties.fields]}")
print(f"Attachments enabled: {target_layer.properties.hasAttachments}")

Folder: Plant ID AI Exercise
Creating service: Plant_Identification_AI_202604211705
  Spatial reference: Web Mercator (102100) — Shape__Area/Length in meters
Created: Plant Identification - AI  (Item ID: ba63860ddf264ff0910ae5cffaab6500)
Folder:  Plant ID AI Exercise
Schema applied: {'success': True, 'layers': [{'name': 'Plant_Observations', 'id': 0}]}
Editor tracking enabled — created/edited fields auto-populated by the server
Waiting for service propagation...
Layer verified — 0 existing features

Layer URL: https://services.arcgis.com/LG9Yn2oFqZi5PnO5/arcgis/rest/services/Plant_Identification_AI_202604211705/FeatureServer/0
Fields:    ['OBJECTID', 'GlobalID', 'common_name', 'latin_name', 'family', 'plant_type', 'condition', 'height_m', 'canopy_width_m', 'observation_date', 'observer', 'notes', 'review_status', 'reviewed_by', 'review_date', 'revision_notes', 'created_user', 'created_date', 'last_edited_user', 'last_edited_date']
Attachments enabled: True


## Step 5 – Deduplicate & Seed Geometries (if source provided)

If a geometry source was loaded in Step 3, the features are **deduplicated** (centroids within 2 m are merged) and then seeded into the new layer as **point features**. Polygon sources are converted to centroids. Only the geometry (+ `canopy_width_m` if available) is transferred — all other fields remain null for the AI service to populate later.

In [11]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from arcgis.features import FeatureLayer
from qaqc.dedup import centroid, deduplicate_features

DEDUP_DISTANCE_M = 2.0  # metres — adjust as needed

# Re-fetch target_layer to ensure a fresh, valid reference
target_layer = FeatureLayer(target_layer.url, gis)
_fc = target_layer.query(where="1=1", return_count_only=True)
print(f"Target layer ready — {_fc} existing features")
print(f"URL: {target_layer.url}\n")

if source_features:
    # Deduplicate source features against themselves (no existing features yet)
    print(f"Deduplicating {len(source_features)} source features (threshold: {DEDUP_DISTANCE_M} m)...")
    unique_features, dup_features, dedup_idx = deduplicate_features(
        new_features=source_features,
        existing_centroids=[],
        distance_m=DEDUP_DISTANCE_M,
        wkid=102100,  # Web Mercator — features are in metres
    )
    print(f"  {dedup_idx.summary()}")

    # Build plain dicts — bypasses arcgis.features.Feature serialization issues
    # where geometry Point objects inject extra keys the REST API rejects.
    seed_dicts = []
    for feat in unique_features:
        geom = feat.geometry
        _x = getattr(geom, "x", None)
        if _x is None and isinstance(geom, dict):
            _x = geom.get("x")
        _y = getattr(geom, "y", None)
        if _y is None and isinstance(geom, dict):
            _y = geom.get("y")

        if _x is not None and _y is not None:
            pt = {"x": float(_x), "y": float(_y),
                  "spatialReference": {"wkid": 102100, "latestWkid": 3857}}
        else:
            g = geom if isinstance(geom, dict) else dict(geom)
            cx, cy = centroid(g)
            pt = {"x": float(cx), "y": float(cy),
                  "spatialReference": {"wkid": 102100, "latestWkid": 3857}}

        seed_dicts.append({"geometry": pt, "attributes": {}})

    # Verify first feature serializes cleanly
    print(f"Sample feature: {json.dumps(seed_dicts[0])}")

    # Add in batches of 500 — pass JSON string directly to avoid Feature serialization
    BATCH = 500
    total_added = 0
    total_features = len(seed_dicts)
    edit_url = f"{target_layer.url}/addFeatures"

    print(f"\nSeeding {total_features} point features into {layer_title}...")
    for i in range(0, total_features, BATCH):
        batch = seed_dicts[i : i + BATCH]
        params = {"f": "json", "features": json.dumps(batch)}
        result = gis._con.post(edit_url, params)
        added = sum(1 for r in result.get("addResults", []) if r.get("success"))
        total_added += added
        print(f"  Batch {i // BATCH + 1}: {added}/{len(batch)} added")

    print(f"\nTotal features seeded: {total_added}/{total_features}")
    if dup_features:
        print(f"Duplicates removed: {len(dup_features)}")
else:
    print("No geometry source — layer is empty, ready for Field Maps collection.")

Target layer ready — 0 existing features
URL: https://services.arcgis.com/LG9Yn2oFqZi5PnO5/arcgis/rest/services/Plant_Identification_AI_202604211705/FeatureServer/0

Deduplicating 1000 source features (threshold: 2.0 m)...
Projected SR (WKID 102100): dedup distance = 2.0 map units
  Checked: 1000  |  Duplicates: 0  |  Unique: 1000
Sample feature: {"geometry": {"x": 488207.23080048454, "y": 6814186.562241104, "spatialReference": {"wkid": 102100, "latestWkid": 3857}}, "attributes": {}}

Seeding 1000 point features into Plant Identification - AI...
  Batch 1: 500/500 added
  Batch 2: 500/500 added

Total features seeded: 1000/1000


## Step 6 – Display the Layer

The new layer is rendered on an interactive map. If geometries were seeded, you'll see them on the map. Otherwise, the map shows the empty layer.

In [12]:
result_map = gis.map()
result_map.basemap.basemap = "satellite"
result_map.content.add(flc_item)
result_map.extent = dict(target_layer.properties.extent)
result_map

Map()

In [13]:
# Summary
fc = target_layer.query(return_count_only=True)
print(f"Layer:       {layer_title}")
print(f"Item ID:     {flc_item.id}")
print(f"URL:         {target_layer.url}")
print(f"Features:    {fc}")
print(f"Schema:      {schema_label}")
print(f"Geometry:    {geom_type}")
print(f"Attachments: Enabled")
print(f"Approval:    review_status = NULL (Arcade sets 'Pending Review' on save)")
print()
print("Next steps:")
print("  1. Run manage-views-and-webmaps.ipynb to create 3 view layers + web maps")
print("  2. Configure popup field visibility in the web maps")
print("  3. Configure Arcade on the Collector view: review_status = 'Pending Review'")
print("  4. Configure feature templates manually in Field Maps")
print("  5. Run the AI enrichment notebook to populate schema fields from photos")
print("  6. Review and approve records in the Approver web map")

flc_item

Layer:       Plant Identification - AI
Item ID:     ba63860ddf264ff0910ae5cffaab6500
URL:         https://services.arcgis.com/LG9Yn2oFqZi5PnO5/arcgis/rest/services/Plant_Identification_AI_202604211705/FeatureServer/0
Features:    1000
Schema:      Plant Identification
Geometry:    esriGeometryPoint
Attachments: Enabled
Approval:    review_status = NULL (Arcade sets 'Pending Review' on save)

Next steps:
  1. Run manage-views-and-webmaps.ipynb to create 3 view layers + web maps
  2. Configure popup field visibility in the web maps
  3. Configure Arcade on the Collector view: review_status = 'Pending Review'
  4. Configure feature templates manually in Field Maps
  5. Run the AI enrichment notebook to populate schema fields from photos
  6. Review and approve records in the Approver web map


<Item title:"Plant Identification - AI" type:Feature Layer Collection owner:ralouta.aiddev>

In [ ]:
import json
from pathlib import Path

# Save item registry — staging uses a separate file so production is untouched
if STAGING:
    _registry_path = Path("..") / "item_registry_staging.json"
else:
    _registry_path = Path("..") / "item_registry.json"

# Load existing registry if present (may have been created by a previous run)
if _registry_path.exists():
    registry = json.loads(_registry_path.read_text())
    print(f"Loaded existing registry from {_registry_path.resolve()}")
else:
    registry = {}

registry["base_layer"] = {
    "item_id": flc_item.id,
    "title": flc_item.title,
    "url": target_layer.url,
    "schema": schema_label,
}

_registry_path.write_text(json.dumps(registry, indent=2))
print(f"Saved item registry to {_registry_path.resolve()}")
print(json.dumps(registry, indent=2))

Loaded existing registry from /Users/rami8629/Library/CloudStorage/OneDrive-Esri/Demos & Blogs/ArcGIS Resources/ArcGIS as Geospatial AI Platofrm/Demos/2026/arcgis-ai-analysis/item_registry.json
Saved item registry to /Users/rami8629/Library/CloudStorage/OneDrive-Esri/Demos & Blogs/ArcGIS Resources/ArcGIS as Geospatial AI Platofrm/Demos/2026/arcgis-ai-analysis/item_registry.json
{
  "base_layer": {
    "item_id": "ba63860ddf264ff0910ae5cffaab6500",
    "title": "Plant Identification - AI",
    "url": "https://services.arcgis.com/LG9Yn2oFqZi5PnO5/arcgis/rest/services/Plant_Identification_AI_202604211705/FeatureServer/0",
    "schema": "Plant Identification"
  }
}


## Next Steps

- **Views & Web Maps**: Run `manage-views-and-webmaps.ipynb` to create 3 view layers (Collector, Approver, Public) and their web maps with popup configuration.
- **Arcade expression**: On the **Collector** view, add a Field Maps calculation on `review_status` → `'Pending Review'` so every save auto-submits the feature for review.
- **Feature Templates**: Configure collection templates manually in Field Maps.
- **Field Maps**: Add the Collector Web Map to Field Maps. Observers will see pre-placed geometries and can attach photos per feature.
- **AI Enrichment**: Use `plant-identification-ai.ipynb` to read attachments and auto-populate schema fields from the Approver view (Pending Review features).
- **Approval Workflow**: Use the Approver web map to approve, reject, or send back records.
- **Editor Tracking**: `created_user`, `created_date`, `last_edited_user`, and `last_edited_date` are auto-populated by ArcGIS Online based on the authenticated user.
- **Add more schemas**: Create a new module in `schemas/` (e.g. `schemas/building_inspection.py`) and register it in `schemas/__init__.py`.